In [ ]:
import requests
from dotenv import load_dotenv
import pandas as pd
import os
import time
import json
import kagglehub
from newsapi import NewsApiClient
from datetime import datetime, timedelta

c:\Users\teo-y\OneDrive\Desktop\DSA4265\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
ALPHA_KEY = os.getenv('ALPHA_VANTAGE_KEY')
NEWSAPI_KEY = os.getenv('NEWSAPI_KEY')

True

## Extraction

### alphavantage

In [4]:
# dates are in YYYYMMDDTHHMM

def extract_news(start_date, end_date, ticker, limit, topics):
    d = {}

    for tick in ticker:
        for topic in topics:
            time.sleep(10)
            url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&topics={topic}&tickers={tick}&limit={limit}&time_from={start_date}&time_to={end_date}&apikey=KCSQ8AKZX0XL0QZT'
            res = requests.get(url)
            data = res.json()

            d[(tick, topic)] = data
    
    return d

In [ ]:
start_date = '20220101T0000'
end_date = '20230101T0000'
tickers = [
    'CL',
    'USO',
    'NG',
    'CRAK',
    'LNG'
]
limit = 10000
topics = [
    'energy_transportation',
    'economy_macro',
    'economy_monetary',
    'economy_fiscal'
]

alphavantage_news = extract_news(start_date, end_date, tickers, limit, topics)

In [32]:
df_rows = []

for (ticker, topic), res in alphavantage_news.items():
    try:
        for each_news in res['feed']:
            time_published = each_news['time_published']
            news_headline = each_news['title']
            url = each_news['url']
            summary = each_news['summary']
            overall_sentiment_score = each_news['overall_sentiment_score']
            overall_sentiment_label = each_news['overall_sentiment_label']

            df_rows.append([ticker, topic, time_published, news_headline, url, summary, overall_sentiment_score, overall_sentiment_label])
    except KeyError:
        print((ticker, topic))

alphavantage_df = pd.DataFrame(df_rows, columns = ['ticker', 'topic', 'time_published', 'news_headline', 'url', 'summary', 'overall_sentiment_score', 'overall_sentiment_label'])

('LNG', 'economy_macro')
('LNG', 'economy_monetary')
('LNG', 'economy_fiscal')


In [33]:
alphavantage_df

,ticker,topic,time_published,news_headline,url,summary,overall_sentiment_score,overall_sentiment_label
0,CL,energy_transportation,20220614T120000,"Republic Cement, Colgate-Palmolive Philippines...",https://www.philstar.com/business/biz-memos/20...,Republic Cement and Colgate-Palmolive Philippi...,0.402010,Bullish
1,CL,economy_macro,20221228T050818,"BNY Mellon Dismissed, Alight, Colgate-Palmoliv...",https://www.planadviser.com/bny-mellon-dismiss...,A U.S. District Court judge dismissed BNY Mell...,-0.081779,Neutral
2,CL,economy_macro,20221227T180652,Procter & Gamble continues sponsoring Russia’s...,https://euromaidanpress.com/2022/12/27/procter...,Procter & Gamble continues to operate two larg...,-0.514546,Bearish
3,CL,economy_macro,20221128T060750,Something to smile about: improving the sustai...,https://mitsloan.mit.edu/action-learning/somet...,Colgate-Palmolive collaborated with MIT Sloan ...,0.411649,Bullish
4,CL,economy_macro,20221104T060750,Indian firms outpace parent companies in valua...,https://www.business-standard.com/article/comp...,Indian companies are experiencing faster valua...,0.323180,Somewhat-Bullish
...,...,...,...,...,...,...,...,...
87,LNG,energy_transportation,20220121T124700,Lawsuit alleges Tellurian chief defrauded inve...,https://www.reuters.com/business/lawsuit-alleg...,A California investor has filed a breach of co...,0.016339,Neutral
88,LNG,energy_transportation,20220111T153257,"U.S. Becomes the Largest LNG Exporter, Aids Eu...",https://www.instituteforenergyresearch.org/int...,The United States became the largest exporter ...,0.177764,Somewhat-Bullish
89,LNG,energy_transportation,20220106T154500,"U.S. Could Pass Australia, Qatar as Top LNG Ex...",https://naturalgasintel.com/news/us-could-pass...,The United States is projected to become the w...,0.168490,Somewhat-Bullish
90,LNG,energy_transportation,20220105T153257,We’re #1! U.S. Ends 2021 As World’s Largest LN...,https://www.energyindepth.org/were-1-u-s-ends-...,The United States became the world's largest e...,0.003245,Neutral


In [ ]:
alphavantage_df.drop_duplicates(subset = ['news_headline', 'url'], inplace = True)
alphavantage_df.to_csv(r'C:\Users\teo-y\OneDrive\Desktop\NUS\DSA4265\group project\data\alphavantage_df.csv')

In [37]:
d = {}

for k, v in alphavantage_news.items():
    d[str(k)] = v

In [40]:
with open(r'C:\Users\teo-y\OneDrive\Desktop\NUS\DSA4265\group project\data\alphavantage_raw_results.json', 'w') as f:
    json.dump(d, f, indent = 4)

### kaggle news dataset

In [8]:
path = kagglehub.dataset_download("rmisra/news-category-dataset")
file_path = os.path.join(path, "News_Category_Dataset_v3.json")
kaggle_news = pd.read_json(file_path, lines=True)

In [16]:
commodities_categories = [
    'WORLD NEWS',
    'POLITICS',
    'ENVIRONMENT',
    'BUSINESS'
]

filters = (
    kaggle_news['category'].isin(commodities_categories) &
    (kaggle_news['date'] >= '2022-01-01')
)
filtered_kaggle_news = kaggle_news[filters]

In [17]:
filtered_kaggle_news

,link,headline,category,short_description,authors,date
7,https://www.huffpost.com/entry/puerto-rico-wat...,Puerto Ricans Desperate For Water After Hurric...,WORLD NEWS,More than half a million people remained witho...,"DÁNICA COTO, AP",2022-09-22
9,https://www.huffpost.com/entry/biden-un-russia...,Biden At UN To Call Russian War An Affront To ...,WORLD NEWS,White House officials say the crux of the pres...,"Aamer Madhani, AP",2022-09-21
10,https://www.huffpost.com/entry/bc-soc-wcup-cap...,World Cup Captains Want To Wear Rainbow Armban...,WORLD NEWS,FIFA has come under pressure from several Euro...,"GRAHAM DUNBAR, AP",2022-09-21
11,https://www.huffpost.com/entry/man-sets-fire-p...,Man Sets Himself On Fire In Apparent Protest O...,WORLD NEWS,The incident underscores a growing wave of pro...,"Mari Yamaguchi, AP",2022-09-21
12,https://www.huffpost.com/entry/fiona-threatens...,Fiona Threatens To Become Category 4 Storm Hea...,WORLD NEWS,Hurricane Fiona lashed the Turks and Caicos Is...,"Dánica Coto, AP",2022-09-21
...,...,...,...,...,...,...
1390,https://www.huffpost.com/entry/marjorie-taylor...,Marjorie Taylor Greene's Personal Twitter Acco...,POLITICS,The Georgia Republican was given the boot afte...,Nina Golgowski,2022-01-02
1391,https://www.huffpost.com/entry/bernard-kerik-p...,Chilling Trump Letter Calling For 'Seizure' Of...,POLITICS,The letter was created a day before Trump disc...,Mary Papenfuss,2022-01-02
1393,https://www.huffpost.com/entry/alexandria-ocas...,Rep. Alexandria Ocasio-Cortez Rips 'Creepy Wei...,POLITICS,The congresswoman fired back at Republican Ste...,Mary Papenfuss,2022-01-01
1394,https://www.huffpost.com/entry/ap-eu-austria-o...,Austrian Holocaust Survivor 'Mrs. Gertrude' Di...,WORLD NEWS,Gertrude Pressburger became famous during Aust...,"Kirsten Grieshaber, AP",2022-01-01


In [18]:
filtered_kaggle_news.to_csv(r'C:\Users\teo-y\OneDrive\Desktop\DSA4265\group project\data\filtered_kaggle_news.csv')

### newsapi

In [46]:
# Init
newsapi = NewsApiClient(api_key=NEWSAPI_KEY)

# --- Config ---
QUERY = (
    '"crude oil" OR "diesel" OR "LNG" OR "liquefied natural gas"'
)
DAYS_BACK = 30

# --- Fetch ---
from_date = (datetime.today() - timedelta(days=DAYS_BACK)).strftime('%Y-%m-%d')
to_date = datetime.today().strftime('%Y-%m-%d')

response = newsapi.get_everything(
    q=QUERY,
    from_param=from_date,
    to=to_date,
    language='en',
    sort_by='publishedAt',
    page_size=100,
    domains='reuters.com, bloomberg.com, ft.com, oilprice.com, wsj.com'
)

# --- Save ---
articles = response['articles']
df = pd.DataFrame(articles)

df['source'] = df['source'].apply(lambda x: x.get('name') if isinstance(x, dict) else x)
df = df[['title', 'source', 'publishedAt', 'description', 'content', 'url']]
df['publishedAt'] = pd.to_datetime(df['publishedAt']).dt.date

print(f"Articles fetched: {len(df)}")
print(df.head())

df.to_csv('oil_diesel_lng_news.csv', index=False)

Articles fetched: 100
                                               title        source  \
0  Brent Hits $118 as Hormuz Shock Blows Out Spre...  OilPrice.com   
1  U.S. Sets April 6 Deadline for Iran as Oil Mar...  OilPrice.com   
2  Asia Burns More Coal as Middle East War Sends ...  OilPrice.com   
3  UAE Investment Firm Buys U.S. Midstream Gas As...  OilPrice.com   
4  India Says 28 Oil and Gas Ships Are Stranded N...  OilPrice.com   

  publishedAt                                        description  \
0  2026-03-31  Brent crude jumped to $118.2 per barrel on Tue...   
1  2026-03-31  US President Donald Trump has renewed his warn...   
2  2026-03-31  Coal is back with a bang in Asia’s power gener...   
3  2026-03-31  2PointZero, an Abu Dhabi-based investment comp...   
4  2026-03-31  At least 28 ships, including vessels carrying ...   

                                             content  \
0  The military conflict and resulting…\r\nChina ...   
1  The United States energy grid…\r\

In [48]:
df.to_csv(r'C:\Users\teo-y\OneDrive\Desktop\DSA4265\group project\data\newsapi_dataset.csv')

## clean and merging 

In [ ]:
alphavantage_df = pd.read_csv(r'C:\Users\teo-y\OneDrive\Desktop\DSA4265\group project\data\alphavantage_df.csv')
kaggle_news = pd.read_csv(r'C:\Users\teo-y\OneDrive\Desktop\DSA4265\group project\data\filtered_kaggle_news.csv')
newsapi_df = pd.read_csv(r'C:\Users\teo-y\OneDrive\Desktop\DSA4265\group project\data\newsapi_dataset.csv')

# date, title, description/summary, relevant_commodities (list of crude oil, disel or LNG), news_category (pre-defined), risk_category (pre-defined)

### helper functions

In [41]:
import re
from datetime import datetime

ticker_mapping = {
    'CL': 'crude oil',
    'USO': 'crude oil',
    'NG': 'LNG',
    'CRAK': 'diesel'
}

# --- Keyword dictionary ---
commodity_keywords = {
    "crude oil": [
        "crude oil", "oil prices", "oil market", "brent", "wti",
        "west texas intermediate", "opec", "opec+",
        "oil production", "oil supply", "oil demand",
        "barrel", "bpd", "petroleum", "oil futures"
    ],
    "diesel": [
        "diesel", "gasoil", "ultra low sulfur diesel", "ulsd",
        "heating oil", "distillate", "middle distillate",
        "diesel prices", "diesel demand", "diesel supply",
        "refining margins", "crack spread"
    ],
    "LNG": [
        "lng", "liquefied natural gas", "natural gas",
        "gas prices", "ttf", "henry hub", "jkm",
        "gas supply", "gas demand", "gas storage",
        "regasification", "lng cargo", "lng export", "lng import"
    ]
}

# --- Precompile regex patterns (faster & safer) ---
commodity_patterns = {
    k: re.compile(r"\b(" + "|".join(map(re.escape, v)) + r")\b", re.IGNORECASE)
    for k, v in commodity_keywords.items()
}

# --- Function to detect commodities ---
def detect_commodities(text):
    matches = []
    for commodity, pattern in commodity_patterns.items():
        if pattern.search(text):
            matches.append(commodity)
    return ",".join(matches)

def normalize_date(date):
    dt = datetime.strptime(date, "%Y%m%dT%H%M%S")
    formatted = dt.strftime("%Y-%m-%d")

    return formatted

### alphavantage

In [42]:
alphavantage_df['ticker'] = alphavantage_df['ticker'].replace(ticker_mapping)
df_1 = alphavantage_df[['time_published', 'news_headline', 'summary', 'ticker']]
df_1.rename(columns = {
    'time_published': 'date',
    'news_headline': 'title',
    'summary': 'description',
    'ticker': 'commodity'
}, inplace = True)

df_1['date'] = df_1['date'].apply(normalize_date)

C:\Users\teo-y\AppData\Local\Temp\ipykernel_3520\1825551537.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1.rename(columns = {
C:\Users\teo-y\AppData\Local\Temp\ipykernel_3520\1825551537.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_1['date'] = df_1['date'].apply(normalize_date)


In [43]:
df_1

,date,title,description,commodity
0,2022-06-14,"Republic Cement, Colgate-Palmolive Philippines...",Republic Cement and Colgate-Palmolive Philippi...,crude oil
1,2022-12-28,"BNY Mellon Dismissed, Alight, Colgate-Palmoliv...",A U.S. District Court judge dismissed BNY Mell...,crude oil
2,2022-12-27,Procter & Gamble continues sponsoring Russia’s...,Procter & Gamble continues to operate two larg...,crude oil
3,2022-11-28,Something to smile about: improving the sustai...,Colgate-Palmolive collaborated with MIT Sloan ...,crude oil
4,2022-11-04,Indian firms outpace parent companies in valua...,Indian companies are experiencing faster valua...,crude oil
...,...,...,...,...
75,2022-01-21,Lawsuit alleges Tellurian chief defrauded inve...,A California investor has filed a breach of co...,LNG
76,2022-01-11,"U.S. Becomes the Largest LNG Exporter, Aids Eu...",The United States became the largest exporter ...,LNG
77,2022-01-06,"U.S. Could Pass Australia, Qatar as Top LNG Ex...",The United States is projected to become the w...,LNG
78,2022-01-05,We’re #1! U.S. Ends 2021 As World’s Largest LN...,The United States became the world's largest e...,LNG


### kaggle news

In [44]:
kaggle_news["commodity"] = kaggle_news["short_description"].apply(lambda x: detect_commodities(str(x)))
df_2 = kaggle_news[['date', 'headline', 'short_description', 'commodity']]
df_2.rename(columns = {
    'headline': 'title',
    'short_description': 'description',
    'commodities': 'commodity'
}, inplace = True)

C:\Users\teo-y\AppData\Local\Temp\ipykernel_3520\107103439.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2.rename(columns = {


In [45]:
df_2

,date,title,description,commodity
0,2022-09-22,Puerto Ricans Desperate For Water After Hurric...,More than half a million people remained witho...,
1,2022-09-21,Biden At UN To Call Russian War An Affront To ...,White House officials say the crux of the pres...,
2,2022-09-21,World Cup Captains Want To Wear Rainbow Armban...,FIFA has come under pressure from several Euro...,
3,2022-09-21,Man Sets Himself On Fire In Apparent Protest O...,The incident underscores a growing wave of pro...,
4,2022-09-21,Fiona Threatens To Become Category 4 Storm Hea...,Hurricane Fiona lashed the Turks and Caicos Is...,
...,...,...,...,...
636,2022-01-02,Marjorie Taylor Greene's Personal Twitter Acco...,The Georgia Republican was given the boot afte...,
637,2022-01-02,Chilling Trump Letter Calling For 'Seizure' Of...,The letter was created a day before Trump disc...,
638,2022-01-01,Rep. Alexandria Ocasio-Cortez Rips 'Creepy Wei...,The congresswoman fired back at Republican Ste...,
639,2022-01-01,Austrian Holocaust Survivor 'Mrs. Gertrude' Di...,Gertrude Pressburger became famous during Aust...,


### newsapi

In [46]:
newsapi_df["commodity"] = newsapi_df["description"].apply(lambda x: detect_commodities(str(x)))
df_3 = newsapi_df[['publishedAt', 'title', 'description', 'commodity']]
df_3.rename(columns = {
    'publishedAt': 'date',
}, inplace = True)

C:\Users\teo-y\AppData\Local\Temp\ipykernel_3520\3492699408.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_3.rename(columns = {


In [47]:
df_3

,date,title,description,commodity
0,2026-03-31,Brent Hits $118 as Hormuz Shock Blows Out Spre...,Brent crude jumped to $118.2 per barrel on Tue...,crude oil
1,2026-03-31,U.S. Sets April 6 Deadline for Iran as Oil Mar...,US President Donald Trump has renewed his warn...,
2,2026-03-31,Asia Burns More Coal as Middle East War Sends ...,Coal is back with a bang in Asia’s power gener...,LNG
3,2026-03-31,UAE Investment Firm Buys U.S. Midstream Gas As...,"2PointZero, an Abu Dhabi-based investment comp...",LNG
4,2026-03-31,India Says 28 Oil and Gas Ships Are Stranded N...,"At least 28 ships, including vessels carrying ...","crude oil,LNG"
...,...,...,...,...
95,2026-03-20,Why the Iran War May Have Just Killed the AI Boom,The stock market spent the first week of the I...,
96,2026-03-20,Why This Energy Shock Will Hit Consumers Harde...,"Arend Kapteyn, the global head of economics an...",
97,2026-03-20,US Drillers Add Oil Rigs For Second Week In A ...,The total number of active drilling rigs for o...,
98,2026-03-20,Oil Prices Ease As US Pulls Out All Stops To S...,"Oil prices pulled back Friday, but not because...",crude oil


### merge

In [49]:
merged_df = pd.concat([df_1, df_2, df_3], axis = 0, ignore_index = True)

In [50]:
merged_df['commodity'].value_counts()

commodity
                    681
crude oil            71
LNG                  57
diesel                7
crude oil,LNG         4
crude oil,diesel      1
Name: count, dtype: int64

In [53]:
merged_df.to_csv(r'C:\Users\teo-y\OneDrive\Desktop\DSA4265\group project\data\merged_df.csv')

In [54]:
merged_df

,date,title,description,commodity
0,2022-06-14,"Republic Cement, Colgate-Palmolive Philippines...",Republic Cement and Colgate-Palmolive Philippi...,crude oil
1,2022-12-28,"BNY Mellon Dismissed, Alight, Colgate-Palmoliv...",A U.S. District Court judge dismissed BNY Mell...,crude oil
2,2022-12-27,Procter & Gamble continues sponsoring Russia’s...,Procter & Gamble continues to operate two larg...,crude oil
3,2022-11-28,Something to smile about: improving the sustai...,Colgate-Palmolive collaborated with MIT Sloan ...,crude oil
4,2022-11-04,Indian firms outpace parent companies in valua...,Indian companies are experiencing faster valua...,crude oil
...,...,...,...,...
816,2026-03-20,Why the Iran War May Have Just Killed the AI Boom,The stock market spent the first week of the I...,
817,2026-03-20,Why This Energy Shock Will Hit Consumers Harde...,"Arend Kapteyn, the global head of economics an...",
818,2026-03-20,US Drillers Add Oil Rigs For Second Week In A ...,The total number of active drilling rigs for o...,
819,2026-03-20,Oil Prices Ease As US Pulls Out All Stops To S...,"Oil prices pulled back Friday, but not because...",crude oil
